# Lab 03: Infrastructure as Code with Stripe & Terraform

**Duration**: ~20 minutes

## Learning Objectives

By the end of this notebook, you will:
- Understand what Infrastructure as Code (IaC) is and why it matters for Stripe configuration
- Know the capabilities of the Stripe Terraform Provider
- Install Terraform in this Colab environment
- Write `.tf` configuration files for products, prices, and customers
- Run `terraform init`, `plan`, and `apply` to create real Stripe resources
- Modify configuration and apply incremental changes

---

## What is Infrastructure as Code?

**Infrastructure as Code (IaC)** manages resources through version-controlled configuration files instead of manual UI clicks.

| Traditional (Dashboard clicks) | Infrastructure as Code |
|--------------------------------|------------------------|
| Manual, error-prone | Automated, consistent |
| Hard to replicate across environments | Identical dev / staging / prod |
| No audit trail | Full Git history |
| Difficult to review | PRs, code review, CI/CD |

## Why Terraform for Stripe?

1. **Version control** — track every change to your product catalog in Git
2. **Reproducibility** — spin up identical Stripe environments for each team or customer
3. **Automation** — integrate with CI/CD pipelines for zero-touch deploys
4. **Disaster recovery** — recreate your entire Stripe setup from code

### Supported Resources

| Terraform resource | Stripe object |
|--------------------|---------------|
| `stripe_product` | Products in your catalog |
| `stripe_price` | One-time or recurring prices |
| `stripe_customer` | Customer records |
| `stripe_coupon` | Discount coupons |
| `stripe_webhook_endpoint` | Webhook configurations |
| `stripe_portal_configuration` | Customer portal settings |

### Workflow

```
Write .tf files  →  terraform plan  →  terraform apply  →  verify in Dashboard
     ↑                  (dry run)        (real changes)           ↓
     └─────────────────── iterate ──────────────────────────────┘
```

---

## Step 1: Install Terraform

Colab runs on Linux, so we download the Terraform binary directly.

In [1]:
%%bash
TERRAFORM_VERSION="1.9.8"
wget -q "https://releases.hashicorp.com/terraform/${TERRAFORM_VERSION}/terraform_${TERRAFORM_VERSION}_linux_amd64.zip" \
     -O /tmp/terraform.zip
unzip -q -o /tmp/terraform.zip -d /usr/local/bin/
terraform version

Terraform v1.9.8
on linux_amd64

Your version of Terraform is out of date! The latest version
is 1.14.8. You can update by downloading from https://www.terraform.io/downloads.html


## Step 2: Configure Your API Key

In [2]:
import os

try:
    from google.colab import userdata
    STRIPE_API_KEY = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    STRIPE_API_KEY = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

# Export to the environment so Terraform can read it
os.environ['STRIPE_API_KEY'] = STRIPE_API_KEY

if 'test' in STRIPE_API_KEY:
    print(f"Safe: Using SANDBOX key ({STRIPE_API_KEY[:12]}...)")
else:
    print("WARNING: This looks like a LIVE key — use sk_test_... for this workshop!")

Loaded API key from Colab Secrets.
Safe: Using SANDBOX key (sk_test_51Rn...)


---

## Step 3: Write the Terraform Configuration Files

We write the `.tf` files directly from Python. In a real project these would live in your Git repository.

### `main.tf` — Provider Configuration

In [3]:
import os

# Create a working directory for our Terraform project
tf_dir = "/tmp/stripe-terraform"
os.makedirs(tf_dir, exist_ok=True)

main_tf = """
terraform {
  required_providers {
    stripe = {
      source  = "stripe/stripe"
      version = "0.1.3"
    }
  }
}

provider "stripe" {
  # Reads STRIPE_API_KEY from the environment automatically
}
"""

with open(f"{tf_dir}/main.tf", "w") as f:
    f.write(main_tf)

print("main.tf written:")
print(main_tf)

main.tf written:

terraform {
  required_providers {
    stripe = {
      source  = "stripe/stripe"
      version = "0.1.3"
    }
  }
}

provider "stripe" {
  # Reads STRIPE_API_KEY from the environment automatically
}



### `products.tf` — Product Catalog

In [4]:
products_tf = """
# Product: Pro Plan
resource "stripe_product" "pro_plan" {
  name        = "Workshop - Pro Plan"
  description = "Professional tier with advanced features"
}

# Price: $29/month
resource "stripe_price" "pro_monthly" {
  product     = stripe_product.pro_plan.id
  currency    = "usd"
  unit_amount = 2900
  recurring {
    interval = "month"
  }
}

# Price: $300/year
resource "stripe_price" "pro_yearly" {
  product     = stripe_product.pro_plan.id
  currency    = "usd"
  unit_amount = 30000
  recurring {
    interval = "year"
  }
}
"""

with open(f"{tf_dir}/products.tf", "w") as f:
    f.write(products_tf)

print("products.tf written:")
print(products_tf)

products.tf written:

# Product: Pro Plan
resource "stripe_product" "pro_plan" {
  name        = "Workshop - Pro Plan"
  description = "Professional tier with advanced features"
}

# Price: $29/month
resource "stripe_price" "pro_monthly" {
  product     = stripe_product.pro_plan.id
  currency    = "usd"
  unit_amount = 2900
  recurring {
    interval = "month"
  }
}

# Price: $300/year
resource "stripe_price" "pro_yearly" {
  product     = stripe_product.pro_plan.id
  currency    = "usd"
  unit_amount = 30000
  recurring {
    interval = "year"
  }
}



### `customers.tf` — Customer Record

In [5]:
customers_tf = """
resource "stripe_customer" "acme" {
  name  = "Workshop Demo Customer"
  email = "demo@acme-corp.example.com"
}
"""

with open(f"{tf_dir}/customers.tf", "w") as f:
    f.write(customers_tf)

print("customers.tf written:")
print(customers_tf)

customers.tf written:

resource "stripe_customer" "acme" {
  name  = "Workshop Demo Customer"
  email = "demo@acme-corp.example.com"
}



### `outputs.tf` — Export Resource IDs

In [6]:
outputs_tf = """
output "price_id_monthly" {
  value       = stripe_price.pro_monthly.id
  description = "Monthly price ID — use this in your Checkout or Payment Links"
}

output "price_id_yearly" {
  value       = stripe_price.pro_yearly.id
  description = "Yearly price ID"
}

output "customer_id" {
  value       = stripe_customer.acme.id
  description = "Demo customer ID"
}
"""

with open(f"{tf_dir}/outputs.tf", "w") as f:
    f.write(outputs_tf)

print("outputs.tf written.")
print(f"\nAll files in {tf_dir}:")
for f in sorted(os.listdir(tf_dir)):
    print(f"  {f}")

outputs.tf written.

All files in /tmp/stripe-terraform:
  customers.tf
  main.tf
  outputs.tf
  products.tf


---

## Step 4: Initialize Terraform

`terraform init` downloads the Stripe provider plugin.

In [7]:
!terraform -chdir=/tmp/stripe-terraform init

Initializing the backend...
Initializing provider plugins...
- Finding stripe/stripe versions matching "0.1.3"...
- Installing stripe/stripe v0.1.3...
- Installed stripe/stripe v0.1.3 (self-signed, key ID A6C87736961A5EF6)
Partner and community providers are signed by their developers.
If you'd like to know more about provider signing, you can read about it here:
https://www.terraform.io/docs/cli/plugins/signing.html
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for your infrastructure. All Terraform commands
should now work.

If you ever set or change modules or backend configuration for Terraform,
rerun this c

You should see `Terraform has been successfully initialized!`

If you see errors:
- Check internet connectivity
- Verify Terraform version is >= 1.0.0 (`terraform version`)

---

## Step 5: Plan — Preview Changes

`terraform plan` is a **dry run**. It shows exactly what will be created, modified, or destroyed without making any changes. Always run this before `apply`.

**Reading the output:**
- `+` = resource will be created
- `-` = resource will be destroyed
- `~` = resource will be updated in place

In [8]:
!terraform -chdir=/tmp/stripe-terraform plan


Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # stripe_customer.acme will be created
  + resource "stripe_customer" "acme" {
      + balance               = (known after apply)
      + business_name         = (known after apply)
      + description           = (known after apply)
      + email                 = "demo@acme-corp.example.com"
      + id                    = (known after apply)
      + individual_name       = (known after apply)
      + invoice_prefix        = (known after apply)
      + metadata              = (known after apply)
      + name                  = "Workshop Demo Customer"
      + next_invoice_sequence = (known after apply)
      + phone                 = (known after apply)
      + tax_exempt            = (known after apply)
      + test_clock            = (known after apply)
    }

  # stripe_price.pro_mo

### Checkpoint

You should see **4 resources to add**:
- `stripe_product.pro_plan`
- `stripe_price.pro_monthly`
- `stripe_price.pro_yearly`
- `stripe_customer.acme`

---

## Step 6: Apply — Create the Resources

In [9]:
!terraform -chdir=/tmp/stripe-terraform apply -auto-approve


Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # stripe_customer.acme will be created
  + resource "stripe_customer" "acme" {
      + balance               = (known after apply)
      + business_name         = (known after apply)
      + description           = (known after apply)
      + email                 = "demo@acme-corp.example.com"
      + id                    = (known after apply)
      + individual_name       = (known after apply)
      + invoice_prefix        = (known after apply)
      + metadata              = (known after apply)
      + name                  = "Workshop Demo Customer"
      + next_invoice_sequence = (known after apply)
      + phone                 = (known after apply)
      + tax_exempt            = (known after apply)
      + test_clock            = (known after apply)
    }

  # stripe_price.pro_mo

### Checkpoint

You should see: `Apply complete! Resources: 4 added, 0 changed, 0 destroyed.`

**Dashboard**: Go to [Products](https://dashboard.stripe.com/test/products) — you should see **Workshop - Pro Plan** with two prices.  
Use the Dashboard search bar and paste a resource ID to jump directly to it.

---

## Step 7: View Outputs

Outputs export resource IDs so your application code (or other Terraform modules) can use them.

In [10]:
!terraform -chdir=/tmp/stripe-terraform output

customer_id = "cus_ULBd7Q7dgDxYvt"
price_id_monthly = "price_1TMVGVBMxfUzotEqMNnlLFJx"
price_id_yearly = "price_1TMVGVBMxfUzotEqmeJLMQpD"


---

## Step 8: Inspect Terraform State

Terraform tracks all managed resources in a **state file**. This is how it knows what already exists and what needs to change.

In [11]:
# List all resources tracked in state
!terraform -chdir=/tmp/stripe-terraform state list

stripe_customer.acme
stripe_price.pro_monthly
stripe_price.pro_yearly
stripe_product.pro_plan


In [12]:
# Inspect a specific resource in state
!terraform -chdir=/tmp/stripe-terraform state show stripe_product.pro_plan

# stripe_product.pro_plan:
resource "stripe_product" "pro_plan" {
    active               = true
    description          = "Professional tier with advanced features"
    id                   = "prod_ULBdaWoJCGXPPE"
    metadata             = {}
    name                 = "Workshop - Pro Plan"
    shippable            = false
    statement_descriptor = null
    type                 = "service"
    unit_label           = null
    url                  = null
}


---

## Exercise: Make a Change and Re-Apply

This is the power of IaC — change the configuration, run `plan`, review the diff, then `apply`.

Let's add a **Starter Plan** product to the catalog.

In [13]:
# Append a new product and price to products.tf
starter_addition = """
# Product: Starter Plan (added in exercise)
resource "stripe_product" "starter_plan" {
  name        = "Workshop - Starter Plan"
  description = "Entry-level tier for small teams"
}

resource "stripe_price" "starter_monthly" {
  product     = stripe_product.starter_plan.id
  currency    = "usd"
  unit_amount = 900  # $9/month
  recurring {
    interval = "month"
  }
}
"""

with open(f"{tf_dir}/products.tf", "a") as f:
    f.write(starter_addition)

print("Starter Plan added to products.tf")

Starter Plan added to products.tf


In [14]:
# Plan shows only the 2 new resources — existing ones are untouched
!terraform -chdir=/tmp/stripe-terraform plan

stripe_customer.acme: Refreshing state... [id=cus_ULBd7Q7dgDxYvt]
stripe_product.pro_plan: Refreshing state... [id=prod_ULBdaWoJCGXPPE]
stripe_price.pro_monthly: Refreshing state... [id=price_1TMVGVBMxfUzotEqMNnlLFJx]
stripe_price.pro_yearly: Refreshing state... [id=price_1TMVGVBMxfUzotEqmeJLMQpD]

Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # stripe_price.starter_monthly will be created
  + resource "stripe_price" "starter_monthly" {
      + active         = true
      + billing_scheme = (known after apply)
      + currency       = "usd"
      + id             = (known after apply)
      + lookup_key     = (known after apply)
      + metadata       = (known after apply)
      + nickname       = (known after apply)
      + product        = (known after apply)
      + tax_behavior   = (known after apply)
      + tiers_mode     = (kno

In [15]:
!terraform -chdir=/tmp/stripe-terraform apply -auto-approve

stripe_product.pro_plan: Refreshing state... [id=prod_ULBdaWoJCGXPPE]
stripe_customer.acme: Refreshing state... [id=cus_ULBd7Q7dgDxYvt]
stripe_price.pro_yearly: Refreshing state... [id=price_1TMVGVBMxfUzotEqmeJLMQpD]
stripe_price.pro_monthly: Refreshing state... [id=price_1TMVGVBMxfUzotEqMNnlLFJx]

Terraform used the selected providers to generate the following execution plan. Resource actions are indicated with the following symbols:
  + create

Terraform will perform the following actions:

  # stripe_price.starter_monthly will be created
  + resource "stripe_price" "starter_monthly" {
      + active         = true
      + billing_scheme = (known after apply)
      + currency       = "usd"
      + id             = (known after apply)
      + lookup_key     = (known after apply)
      + metadata       = (known after apply)
      + nickname       = (known after apply)
      + product        = (known after apply)
      + tax_behavior   = (known after apply)
      + tiers_mode     = (kno

**Dashboard**: Refresh [Products](https://dashboard.stripe.com/test/products) — you should now see both **Workshop - Pro Plan** and **Workshop - Starter Plan**.

---

## Cleanup (Optional)

`terraform destroy` removes all resources managed by this configuration from Stripe.

In [16]:
# Uncomment to delete all Terraform-managed resources from Stripe
# !terraform -chdir=/tmp/stripe-terraform destroy -auto-approve

---

## Summary

| Command | What it does |
|---------|--------------|
| `terraform init` | Downloads the Stripe provider plugin |
| `terraform plan` | Dry run — previews what will change |
| `terraform apply` | Executes the changes against the Stripe API |
| `terraform output` | Shows exported resource IDs |
| `terraform state list` | Lists all tracked resources |
| `terraform destroy` | Deletes all managed resources |

### Key Takeaways

- **Always `plan` before `apply`** — review the diff before making changes
- **State is critical** — Terraform uses the state file to determine what already exists
- **Outputs** export IDs for use in your application or other modules
- **IaC enables CI/CD** — commit `.tf` files, run `terraform apply` in your pipeline
